In [112]:
import json
import re
from typing import Any

type SExpr = str | list[SExpr]

def parse(s: str) -> list[SExpr]:
    stack: list[list[SExpr]] = [[]]
    for token in re.findall("|".join([
        r"[\(\)]",
        r"[\w-]+",
        r"\"(?:\\\"|[^\"])*\"",
        r"\'(?:\\\"|[^\'])*\'",
    ]), s):
        if token == "(":
            child = []
            stack[-1].append(child)
            stack.append(child)
        elif token == ")":
            if len(stack) <= 1:
                raise Exception("unbalanced parentheses")
            stack.pop()
        elif token[0] == '"' or token[0] == "'":
            stack[-1].append(eval(token))
        else:
            stack[-1].append(token)
    if len(stack) != 1:
        raise Exception("unbalanced parentheses")
    return stack[0]

type Value = Any

def evaluate_proc(proc: SExpr, arg: Value) -> Value:
    if isinstance(proc, str):
        if proc == "json":
            return json.loads(arg)
    else:
        if proc[0] == "str":
            return proc[1]
        elif proc[0] == "field":
            assert len(proc) == 2
            assert isinstance(proc[1], str)
            assert isinstance(arg, dict)
            return arg[proc[1]]
        elif proc[0] == "index":
            assert len(proc) == 2
            assert isinstance(proc[1], str)
            assert isinstance(arg, list)
            return arg[int(proc[1])]
        elif proc[0] == "map":
            assert len(proc) == 2
            assert isinstance(proc[1], list)
            assert isinstance(arg, list)
            return [evaluate_procs(proc[1], item) for item in arg]
    raise Exception(f"unknown proc: {proc}")

def evaluate_procs(procs: list[SExpr], arg: Value) -> Value:
    for proc in procs:
        arg = evaluate_proc(proc, arg)
    return arg

def evaluate(s: str) -> Value:
    procs = parse(s)
    return evaluate_procs(procs, None)


arg = json.dumps([
    {"x": [[1], 2]},
    {"x": [[3], 4]}
])
s = rf"""
(str {repr(arg)})
json
(map ((field x) (index 0)))
(map ((index 0)))
"""
print(s)
evaluate(s)


(str '[{"x": [[1], 2]}, {"x": [[3], 4]}]')
json
(map ((field x) (index 0)))
(map ((index 0)))



[1, 3]